## HW10 — линейная бинарная классификация

Обработка категориальных признаков, логистическая регрессия, holdout и L2-регуляризация. Основа — HW_10_.ipynb. Полный набор Malware не запускался; исторические outputs не считаются актуальными.

In [ ]:
from codecs import namereplace_errors

import pandas as pd
import numpy as np
import warnings

# from traitlets.config import catch_config_error

### Задание 1: исходные данные

Используйте Microsoft Malware Detection. Целевая колонка HasDetections содержит 0/1; остальные колонки описывают компьютер.

In [ ]:
data = pd.read_csv('./train.csv',low_memory=False)

In [ ]:
data.head()

Удалите константные колонки, ProductName и MachineIdentifier.

In [ ]:
### Your code is here
data = data.drop(columns=['ProductName', 'MachineIdentifier'])

constant_cols = data.columns[data.nunique(dropna=False) <= 1]
print("Константные признаки:", constant_cols.tolist())

data = data.drop(columns=constant_cols)

In [ ]:
data.columns

Посчитайте число объектов каждого класса.

In [ ]:
print(sum(data['HasDetections'] == 1), '- positive class,')
print(sum(data['HasDetections'] == 0), '- negative class')

Почему при оценке классификации важно знать баланс классов? Собственный письменный ответ в исходнике отсутствует.

### Пропуски

Удалите признаки с долей пропусков больше половины. Числовые значения заполните медианами, категориальные — строкой NaN.

In [ ]:
cols_to_drop = data.columns[data.isna().mean() > 0.5]

print("Признаки с большим количеством пропусков:", cols_to_drop.tolist())

data = data.drop(columns=cols_to_drop)

In [ ]:
### Your code is here

# numeric_columns = data.loc[:,data.dtypes!=np.object_].columns
# data.loc[:,data.dtypes!=np.object_].head(2)

### better to use
numeric_columns = data.select_dtypes(include=[np.number]).columns
data[numeric_columns] = data[numeric_columns].fillna(data[numeric_columns].median())

In [ ]:
# categorical_columns = data.select_dtypes(include=["number","object"]).columns

# Отбираем категориальные колонки строго по логике вашего урока
# categorical_columns = data.loc[:, data.dtypes == object].columns
# Заменяем отсутствующие значения строкой 'NaN' (как написано на вашем скриншоте)
# data[categorical_columns] = data[categorical_columns].fillna(value='NaN')

categorical_columns = data.select_dtypes(
    include=['object', 'string', 'category']
).columns

data[categorical_columns] = data[categorical_columns].fillna('NaN')

Сохраните копию подготовленных данных в data_2 для следующего способа кодирования.

In [ ]:
data_2 = data.copy()

data_2.shape

Закодируйте все категориальные признаки через One-Hot Encoding.

In [ ]:
### Your code is here
# for col in categorical_columns:
#     one_hot = pd.get_dummies(data[col],prefix=col,drop_first=True)
#     data = pd.concat((data.drop(col, axis=1), one_hot), axis=1)
### bad scenario - Unable to allocate 4.35 GiB for an array with shape (200000, 23379) and data type bool

# data = pd.get_dummies(data, columns=categorical_columns, drop_first=True, sparse=True)

existing_cat_cols = [col for col in categorical_columns if col in data.columns]

# Кодируем в одну строчку с флагом sparse=True, чтобы не было MemoryError
data = pd.get_dummies(data, columns=existing_cat_cols, drop_first=True, sparse=True)


# encoded_cats = pd.get_dummies(
#     data[categorical_columns],
#     prefix=categorical_columns,
#     drop_first=True,
#     sparse=True  # КРИТИЧЕСКИ ВАЖНО: сохраняет матрицу в разреженном виде
# )
#
# data = data.drop(categorical_columns, axis=1)
# data = pd.concat([data, encoded_cats], axis=1)

In [ ]:
data.head()

### Отложенная выборка

Используйте holdout в пропорции train:test = 3:1. В этом упражнении достаточно одного разделения.

In [ ]:
from sklearn.model_selection import train_test_split

X = data.drop(columns=['HasDetections'])
y = data['HasDetections']

X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    test_size=0.25,
                                                    shuffle=True,
                                                    random_state=1)

Соберите pipe из StandardScaler и LogisticRegression без регуляризации. В текущем API отсутствие штрафа задаётся penalty=None.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

### Your code is here
# pipe = Pipeline(['scaler':StandardScaler(),'logistic_model':LogisticRegression(penalty='none')])
pipe = Pipeline([('scaler', StandardScaler()),
                 ('LR', LogisticRegression(penalty=None))])

Для оценки accuracy на train и test передайте cross_validate список с одной парой позиционных индексов вашего holdout.

In [ ]:
from sklearn.model_selection import cross_validate
import datetime

custom_cv = [(X_train.index.to_list(), X_test.index.to_list())]

begin_time = datetime.datetime.now()

cv_result_pipe = cross_validate(pipe, X, y, scoring='accuracy',
                                cv=custom_cv, return_train_score=True)


print(f"Accuracy на трейне: {np.mean(cv_result_pipe['train_score']).round(3)}")
print(f"Accuracy на тесте: {np.mean(cv_result_pipe['test_score']).round(3)}")

print(f"Время работы алгоритма: {datetime.datetime.now() - begin_time}")

### Время и число признаков

Сравните время работы и качество. Многочисленные OHE-колонки увеличивают затраты памяти и времени; ограничение max_iter может остановить оптимизацию до сходимости.

### Задание 2: другое кодирование

Снова используйте data_2 и то же разделение. Категории с числом значений больше пяти закодируйте средним таргета, остальные — One-Hot Encoding.

In [ ]:
# Восстанавливаем данные до One-Hot Encoding
data_2_encoded = data_2.copy()

# Получаем актуальные категориальные признаки
categorical_columns_2 = data_2_encoded.select_dtypes(
    include=['object', 'string', 'category']
).columns

for col in categorical_columns_2:

    # Категорий не больше 5 — One-Hot Encoding
    if data_2_encoded[col].nunique() <= 5:
        one_hot = pd.get_dummies(
            data_2_encoded[col],
            prefix=col,
            drop_first=True
        )

        data_2_encoded = pd.concat(
            (data_2_encoded.drop(columns=col), one_hot),
            axis=1
        )

    # Категорий больше 5 — Mean Target Encoding
    else:
        mean_target = data_2_encoded.groupby(col)['HasDetections'].mean()
        data_2_encoded[col] = data_2_encoded[col].map(mean_target)

print(data_2_encoded.shape)

In [ ]:
X_2 = data_2_encoded.drop(columns=['HasDetections'])
y_2 = data_2_encoded['HasDetections']

Обучите ту же модель и сравните скорость и качество с предыдущим вариантом.

In [ ]:
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
import numpy as np
import datetime


train_idx, test_idx = train_test_split(
    np.arange(len(X_2)),
    test_size=0.25,
    shuffle=True,
    random_state=1
)

# Собираем pipeline
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('LR', LogisticRegression(penalty=None, max_iter=1000))
])

# Один fold: train и test
custom_cv = [(train_idx, test_idx)]

begin_time = datetime.datetime.now()

cv_result_pipe = cross_validate(
    pipe,
    X_2,
    y_2,
    scoring='accuracy',
    cv=custom_cv,
    return_train_score=True
)

print(f"Accuracy на трейне: {np.mean(cv_result_pipe['train_score']):.3f}")
print(f"Accuracy на тесте: {np.mean(cv_result_pipe['test_score']):.3f}")
print(f"Время работы алгоритма: {datetime.datetime.now() - begin_time}")

### Задание 3: регуляризация

Используйте L2-регуляризацию LogisticRegression. Допустимо изменить параметр существующего pipe через set_params.

In [ ]:
### Your code is here
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('LR', LogisticRegression(penalty='l2'))
])

Переберите силу регуляризации и сравните качество моделей.

In [ ]:
alphas = np.linspace(0.01, 100, 100)
alphas

Используйте GridSearchCV для выбора параметра из заданной сетки.

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid ={
    'LR__C': alphas
}
pipe.set_params(LR__penalty='l2')
### Your code is here
# gridcv = GridSearchCV(estimator=pipe, param_grid=param_grid)
gridcv = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=2,
    verbose=2
)

# gridcv.fit(X_2, y_2)
gridcv.fit(X_2, y_2)


print("Лучший параметр:", gridcv.best_params_)
print("CV accuracy:", gridcv.best_score_)

### Бонус

Создайте признаки-мономы второй степени и попробуйте получить accuracy не ниже 0.65 на test. Кодирование и регуляризацию можно выбрать самостоятельно. Пустое решение оставлено незаполненным.

In [ ]:
### Your code is here